# LeetCode 1471 — The k Strongest Values in an Array bằng Python

Link đề: https://leetcode.com/problems/the-k-strongest-values-in-an-array/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Sorting

Skeleton LeetCode:

```python
class Solution:
    def getStrongest(self, arr: list[int], k: int) -> list[int]:
        ...
```

## 1. Mục tiêu học

1. Tính đúng **"tâm" (centre)** theo định nghĩa của đề: phần tử ở vị trí `(n − 1) // 2` của mảng **đã sắp xếp** (không phải trung vị trung bình cộng).
2. Biến một quan hệ so sánh "mạnh hơn" có **tie-break** thành **khóa sắp xếp** dạng tuple `(|x − m|, x)`.
3. Dùng **hai con trỏ** trên mảng đã sắp: phần tử mạnh nhất luôn nằm ở **một trong hai đầu**.
4. Biết các công cụ `sorted(key=...)`, `heapq.nlargest`, `functools.cmp_to_key` và khi nào dùng cái nào.
5. Kiểm thử bài "trả về theo thứ tự bất kỳ" bằng cách **so sánh sau khi sắp xếp**.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số `k`. Gọi `m` là **tâm** của mảng: sắp `arr` tăng dần, `m` là phần tử ở chỉ số `(n − 1) // 2` (0-indexed).

Giá trị `arr[i]` **mạnh hơn** `arr[j]` nếu `|arr[i] − m| > |arr[j] − m|`; nếu bằng nhau thì `arr[i]` mạnh hơn khi `arr[i] > arr[j]`.

Trả về **k giá trị mạnh nhất** (thứ tự bất kỳ).

Ví dụ về tâm: `arr = [6, -3, 7, 2, 11]` → sắp `[-3, 2, 6, 7, 11]`, `n = 5`, chỉ số `2` ⇒ `m = 6`. `arr = [-7, 22, 17, 3]` → sắp `[-7, 3, 17, 22]`, chỉ số `(4−1)//2 = 1` ⇒ `m = 3`.

**Ví dụ 1:** `arr = [1,2,3,4,5], k = 2` → `[5,1]` (tâm 3; thứ tự mạnh `[5,1,4,2,3]`; `[1,5]` cũng đúng).

**Ví dụ 2:** `arr = [1,1,3,5,5], k = 2` → `[5,5]`.

**Ví dụ 3:** `arr = [6,7,11,7,6,8], k = 5` → `[11,8,6,6,7]` (tâm 7; mọi hoán vị đều được chấp nhận).

**Ràng buộc:** `1 <= arr.length <= 10^5`, `-10^5 <= arr[i] <= 10^5`, `1 <= k <= arr.length`.

In [1]:
import heapq
import random
import statistics
import time
from functools import cmp_to_key
from typing import List

examples = [
    ([1, 2, 3, 4, 5], 2, [5, 1]),
    ([1, 1, 3, 5, 5], 2, [5, 5]),
    ([6, 7, 11, 7, 6, 8], 5, [11, 8, 6, 6, 7]),
]
for arr, k, want in examples:
    print(f"arr={arr}, k={k} -> một đáp án hợp lệ: {want}")

arr=[1, 2, 3, 4, 5], k=2 -> một đáp án hợp lệ: [5, 1]
arr=[1, 1, 3, 5, 5], k=2 -> một đáp án hợp lệ: [5, 5]
arr=[6, 7, 11, 7, 6, 8], k=5 -> một đáp án hợp lệ: [11, 8, 6, 6, 7]


## 3. Kiến thức nền

### 3.1 Tâm theo định nghĩa của đề

Sắp xếp rồi lấy chỉ số `(n − 1) // 2`. Với `n` chẵn, đây là phần tử **giữa bên trái**, KHÔNG phải trung bình hai phần tử giữa như `statistics.median`.

In [2]:
def centre(arr):
    s = sorted(arr)
    return s[(len(s) - 1) // 2]

for a in ([6, -3, 7, 2, 11], [-7, 22, 17, 3], [1, 2, 3, 4]):
    print(a, "-> tâm =", centre(a), "| statistics.median =", statistics.median(a))

[6, -3, 7, 2, 11] -> tâm = 6 | statistics.median = 6
[-7, 22, 17, 3] -> tâm = 3 | statistics.median = 10.0
[1, 2, 3, 4] -> tâm = 2 | statistics.median = 2.5


### 3.2 `sorted` với `key` là tuple

Python so sánh tuple **từ trái sang phải**: so phần tử đầu, bằng nhau mới so phần tử thứ hai. Vì vậy khóa `(|x − m|, x)` mã hóa đúng quan hệ "mạnh hơn": so độ lệch trước, hòa thì so giá trị. Thêm `reverse=True` để mạnh nhất đứng đầu.

In [3]:
arr, m = [1, 2, 3, 4, 5], 3
print(sorted(arr, key=lambda x: (abs(x - m), x), reverse=True))
print((2, 5) > (2, 1), (2, 1) > (1, 4))

[5, 1, 4, 2, 3]
True True


### 3.3 `functools.cmp_to_key` — sắp theo hàm so sánh

Nếu chỉ có hàm `cmp(a, b)` trả về số âm/0/dương, dùng `cmp_to_key` để đưa vào `sorted`. Cách này gần đề bài nhất nhưng chậm hơn khóa tuple.

In [4]:
def stronger_cmp(m):
    def cmp(a, b):
        da, db = abs(a - m), abs(b - m)
        if da != db:
            return -1 if da > db else 1      # mạnh hơn đứng trước
        return -1 if a > b else (1 if a < b else 0)
    return cmp

print(sorted([1, 2, 3, 4, 5], key=cmp_to_key(stronger_cmp(3))))

[5, 1, 4, 2, 3]


### 3.4 `heapq.nlargest(k, iterable, key=...)`

Lấy `k` phần tử lớn nhất theo khóa trong `O(n log k)` (duy trì heap kích thước `k`), không cần sắp toàn bộ.

In [5]:
print(heapq.nlargest(2, [1, 2, 3, 4, 5], key=lambda x: (abs(x - 3), x)))
print(heapq.nlargest(3, [5, 1, 9, 7, 3]))

[5, 1]
[9, 7, 5]


### 3.5 Hai con trỏ trên mảng đã sắp

Trên mảng tăng dần, giá trị xa tâm nhất phải nằm ở **đầu trái** (rất nhỏ) hoặc **đầu phải** (rất lớn). Con trỏ `i` từ trái, `j` từ phải; mỗi bước lấy đầu mạnh hơn và co đầu đó lại.

In [6]:
s = [1, 2, 3, 4, 5]
i, j = 0, len(s) - 1
print("hai đầu:", s[i], s[j], "| độ lệch so với 3:", abs(s[i] - 3), abs(s[j] - 3))

hai đầu: 1 5 | độ lệch so với 3: 2 2


### 3.6 So sánh đáp án "thứ tự bất kỳ"

Đề chấp nhận mọi hoán vị ⇒ khi test ta so `sorted(got) == sorted(want)`. Với bài này, **tập** (multiset) k phần tử mạnh nhất là **duy nhất** vì quan hệ "mạnh hơn" là thứ tự toàn phần trên giá trị (giá trị bằng nhau thì hoán đổi được).

In [7]:
print(sorted([5, 1]) == sorted([1, 5]), sorted([11, 8, 6, 6, 7]) == sorted([7, 6, 11, 6, 8]))

True True


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 3: `arr = [6,7,11,7,6,8]`, sắp: `[6,6,7,7,8,11]`, `n = 6`, chỉ số `(6−1)//2 = 2` ⇒ `m = 7`.

| x | \|x − 7\| | khóa (độ lệch, x) |
|---|---|---|
| 6 | 1 | (1, 6) |
| 6 | 1 | (1, 6) |
| 7 | 0 | (0, 7) |
| 7 | 0 | (0, 7) |
| 8 | 1 | (1, 8) |
| 11 | 4 | (4, 11) |

Xếp khóa giảm dần: `(4,11) > (1,8) > (1,6) = (1,6) > (0,7) = (0,7)` ⇒ thứ tự mạnh `[11, 8, 6, 6, 7, 7]`, lấy 5 phần tử đầu `[11, 8, 6, 6, 7]`.

Để ý: trong mảng đã sắp, `11` ở đầu phải, `8` kế đó, rồi `6, 6` ở đầu trái — các phần tử mạnh **ăn dần từ hai đầu vào giữa**.

In [8]:
arr = [6, 7, 11, 7, 6, 8]
m = centre(arr)
for x in sorted(arr):
    print(x, abs(x - m), (abs(x - m), x))
print("thứ tự mạnh:", sorted(arr, key=lambda x: (abs(x - m), x), reverse=True))

6 1 (1, 6)
6 1 (1, 6)
7 0 (0, 7)
7 0 (0, 7)
8 1 (1, 8)
11 4 (4, 11)
thứ tự mạnh: [11, 8, 6, 6, 7, 7]


## 5. Cách 1 — Brute Force / baseline: chọn mạnh nhất k lần

**Ý tưởng:** tìm tâm; rồi lặp `k` lần: duyệt toàn bộ phần tử còn lại, chọn phần tử mạnh nhất theo đúng định nghĩa (so độ lệch, hòa thì so giá trị), bỏ nó ra.

Đây là cách dịch thẳng định nghĩa "k phần tử mạnh nhất", dễ tin là đúng.

In [9]:
def get_strongest_brute(arr, k):
    m = centre(arr)
    rest = list(arr)
    ans = []
    for _ in range(k):
        best = 0
        for t in range(1, len(rest)):
            a, b = rest[t], rest[best]
            if abs(a - m) > abs(b - m) or (abs(a - m) == abs(b - m) and a > b):
                best = t
        ans.append(rest.pop(best))
    return ans

for arr, k, want in examples:
    got = get_strongest_brute(arr, k)
    print(got, "| hợp lệ?", sorted(got) == sorted(want))

[5, 1] | hợp lệ? True
[5, 5] | hợp lệ? True
[11, 8, 6, 6, 7] | hợp lệ? True


**Độ phức tạp:** tìm tâm `O(n log n)`; mỗi lần chọn `O(n)` (cộng `pop` `O(n)`) ⇒ `O(n · k)`. Khi `k ≈ n = 10^5`: `10^10` — quá chậm.

**Nút thắt:** mỗi lượt lại quét toàn bộ để tìm "mạnh nhất kế tiếp", lặp lại công việc so sánh. Thay vào đó, **sắp xếp một lần** theo độ mạnh là đủ.

In [10]:
rng = random.Random(1471)
for n in (1000, 2000, 4000):
    arr = [rng.randint(-10**5, 10**5) for _ in range(n)]
    t = time.perf_counter()
    get_strongest_brute(arr, n // 2)
    print(f"n={n}, k={n // 2}: {time.perf_counter() - t:.3f}s")

n=1000, k=500: 0.076s


n=2000, k=1000: 0.231s


n=4000, k=2000: 1.071s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — khóa sắp xếp:** "mạnh hơn" ⇔ tuple `(|x − m|, x)` lớn hơn. Sắp giảm dần theo khóa và lấy `k` đầu tiên.

**Insight 2 — hai đầu:** sau khi sắp `arr` tăng dần, hàm `|x − m|` **giảm** khi đi từ trái vào tới `m` rồi **tăng** khi đi tiếp sang phải (hình chữ V). Nên phần tử mạnh nhất trong đoạn còn lại `[i..j]` luôn là `arr[i]` hoặc `arr[j]`.

**Quy tắc chọn:**
- **Nếu** `|arr[j] − m| >= |arr[i] − m|` **thì** lấy `arr[j]` (khi hòa, `arr[j] >= arr[i]` nên `arr[j]` mạnh hơn hoặc bằng), `j -= 1`.
- **Ngược lại** lấy `arr[i]`, `i += 1`.

**Invariant:** sau `t` bước, các phần tử đã lấy chính là `t` phần tử mạnh nhất, và đoạn `[i..j]` chứa đúng các phần tử còn lại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp theo khóa `(|x − m|, x)` rồi cắt k phần tử

**Từng đoạn:**
1. `arr.sort()` và `m = arr[(n − 1) // 2]`.
2. `sorted(arr, key=lambda x: (abs(x − m), x), reverse=True)[:k]`.

In [11]:
def get_strongest_sort(arr, k):
    s = sorted(arr)
    m = s[(len(s) - 1) // 2]
    return sorted(s, key=lambda x: (abs(x - m), x), reverse=True)[:k]

for arr, k, want in examples:
    got = get_strongest_sort(arr, k)
    print(got, sorted(got) == sorted(want))

[5, 1] True
[5, 5] True
[11, 8, 6, 6, 7] True


Code nộp LeetCode (Cách 2):

In [12]:
class Solution:
    def getStrongest(self, arr: List[int], k: int) -> List[int]:
        arr.sort()
        m = arr[(len(arr) - 1) // 2]
        arr.sort(key=lambda x: (abs(x - m), x), reverse=True)
        return arr[:k]

print([Solution().getStrongest(list(a), k) for a, k, _ in examples])

[[5, 1], [5, 5], [11, 8, 6, 6, 7]]


**Độ phức tạp:** hai lần sắp `O(n log n)`, bộ nhớ `O(n)` cho khóa. Ngắn, rõ, ít lỗi.

### 7.2 Cách 3 — Hai con trỏ trên mảng đã sắp

**Từng đoạn:**
1. Sắp `arr`, lấy `m`.
2. `i = 0`, `j = n − 1`. Lặp `k` lần: so `arr[j] − m` với `m − arr[i]` (đều không âm khi `i ≤ tâm ≤ j`… nhưng dùng `abs` cho an toàn).
3. Dùng `>=` để khi hòa ưu tiên `arr[j]` (lớn hơn).

In [13]:
def get_strongest_two_pointers(arr, k):
    s = sorted(arr)
    m = s[(len(s) - 1) // 2]
    i, j = 0, len(s) - 1
    ans = []
    while len(ans) < k:
        if abs(s[j] - m) >= abs(s[i] - m):
            ans.append(s[j]); j -= 1
        else:
            ans.append(s[i]); i += 1
    return ans

for arr, k, want in examples:
    got = get_strongest_two_pointers(arr, k)
    print(got, sorted(got) == sorted(want))

[5, 1] True
[5, 5] True
[11, 8, 6, 6, 7] True


Code nộp LeetCode (Cách 3):

In [14]:
class Solution:
    def getStrongest(self, arr: List[int], k: int) -> List[int]:
        arr.sort()
        m = arr[(len(arr) - 1) // 2]
        i, j = 0, len(arr) - 1
        ans = []
        for _ in range(k):
            if arr[j] - m >= m - arr[i]:
                ans.append(arr[j]); j -= 1
            else:
                ans.append(arr[i]); i += 1
        return ans

print([Solution().getStrongest(list(a), k) for a, k, _ in examples])

[[5, 1], [5, 5], [11, 8, 6, 6, 7]]


Trong code nộp ta viết `arr[j] − m >= m − arr[i]` thay cho `abs`: chừng nào còn lấy (tối đa `n` phần tử), nếu `j` đã lùi qua tâm thì mọi phần tử bên phải tâm đều đã lấy và ngược lại; ta chứng minh kỹ ở mục 10 và kiểm chứng bằng test ngẫu nhiên.

**Độ phức tạp:** sắp `O(n log n)` + `O(k)` hai con trỏ. Bộ nhớ `O(k)` cho đáp án (ngoài sort). Chỉ cần **một lần sắp** thay vì hai.

### 7.3 Cách 4 — Quickselect tìm tâm + `heapq.nlargest`

Không cần sắp toàn bộ:
1. Tìm phần tử nhỏ thứ `(n − 1) // 2` bằng **quickselect** (trung bình `O(n)`).
2. `heapq.nlargest(k, arr, key=lambda x: (abs(x − m), x))` — `O(n log k)`.

Có lợi khi `k` nhỏ so với `n`.

In [15]:
def quickselect(a, idx, rng=random.Random(0)):
    a = list(a)
    lo, hi = 0, len(a) - 1
    while True:
        if lo == hi:
            return a[lo]
        pivot = a[rng.randint(lo, hi)]
        lt = [x for x in a[lo:hi + 1] if x < pivot]
        eq = [x for x in a[lo:hi + 1] if x == pivot]
        gt = [x for x in a[lo:hi + 1] if x > pivot]
        a[lo:hi + 1] = lt + eq + gt
        if idx < lo + len(lt):
            hi = lo + len(lt) - 1
        elif idx < lo + len(lt) + len(eq):
            return pivot
        else:
            lo = lo + len(lt) + len(eq)

def get_strongest_heap(arr, k):
    m = quickselect(arr, (len(arr) - 1) // 2)
    return heapq.nlargest(k, arr, key=lambda x: (abs(x - m), x))

for arr, k, want in examples:
    got = get_strongest_heap(arr, k)
    print(got, sorted(got) == sorted(want))

[5, 1] True
[5, 5] True
[11, 8, 6, 6, 7] True


Code nộp LeetCode (Cách 4) — dùng `sorted` cho tâm cho gọn, phần chọn k dùng heap:

In [16]:
class Solution:
    def getStrongest(self, arr: List[int], k: int) -> List[int]:
        m = sorted(arr)[(len(arr) - 1) // 2]
        return heapq.nlargest(k, arr, key=lambda x: (abs(x - m), x))

print([Solution().getStrongest(list(a), k) for a, k, _ in examples])

[[5, 1], [5, 5], [11, 8, 6, 6, 7]]


**Độ phức tạp:** quickselect trung bình `O(n)` (xấu nhất `O(n²)` nhưng pivot ngẫu nhiên làm điều đó cực hiếm) + `O(n log k)`. Trong Python thực tế `sorted` (viết bằng C) thường nhanh hơn quickselect thuần Python.

In [17]:
rng = random.Random(3)
big = [rng.randint(-10**5, 10**5) for _ in range(10**5)]
for name, f in [("sort theo khóa", get_strongest_sort), ("hai con trỏ", get_strongest_two_pointers),
                ("quickselect+heap", get_strongest_heap)]:
    for k in (10, 50000):
        t = time.perf_counter()
        f(big, k)
        print(f"{name:17s} k={k:5d}: {time.perf_counter() - t:.3f}s")

sort theo khóa    k=   10: 0.063s


sort theo khóa    k=50000: 0.065s
hai con trỏ       k=   10: 0.019s


hai con trỏ       k=50000: 0.028s
quickselect+heap  k=   10: 0.043s


quickselect+heap  k=50000: 0.162s


## 8. TRACE từng bước

Trace Cách 3 với `arr = [6,7,11,7,6,8]`, `k = 5`. Sắp: `[6,6,7,7,8,11]`, `m = 7`.

| bước | i | j | arr[i] | arr[j] | lệch trái | lệch phải | lấy |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 5 | 6 | 11 | 1 | 4 | 11 (phải) |
| 2 | 0 | 4 | 6 | 8 | 1 | 1 | 8 (hòa → phải) |
| 3 | 0 | 3 | 6 | 7 | 1 | 0 | 6 (trái) |
| 4 | 1 | 3 | 6 | 7 | 1 | 0 | 6 (trái) |
| 5 | 2 | 3 | 7 | 7 | 0 | 0 | 7 (hòa → phải) |

In [18]:
def trace_two_pointers(arr, k):
    s = sorted(arr)
    m = s[(len(s) - 1) // 2]
    print(f"sắp: {s}, tâm m = s[{(len(s) - 1) // 2}] = {m}")
    i, j = 0, len(s) - 1
    ans = []
    for step in range(1, k + 1):
        dl, dr = abs(s[i] - m), abs(s[j] - m)
        if dr >= dl:
            pick, side = s[j], "phải" + (" (hòa)" if dr == dl else "")
            j -= 1
        else:
            pick, side = s[i], "trái"
            i += 1
        ans.append(pick)
        print(f"bước {step}: lệch trái={dl}, lệch phải={dr} -> lấy {pick} ({side}); i={i}, j={j}, ans={ans}")
    return ans

_ = trace_two_pointers([6, 7, 11, 7, 6, 8], 5)

sắp: [6, 6, 7, 7, 8, 11], tâm m = s[2] = 7
bước 1: lệch trái=1, lệch phải=4 -> lấy 11 (phải); i=0, j=4, ans=[11]
bước 2: lệch trái=1, lệch phải=1 -> lấy 8 (phải (hòa)); i=0, j=3, ans=[11, 8]
bước 3: lệch trái=1, lệch phải=0 -> lấy 6 (trái); i=1, j=3, ans=[11, 8, 6]
bước 4: lệch trái=1, lệch phải=0 -> lấy 6 (trái); i=2, j=3, ans=[11, 8, 6, 6]
bước 5: lệch trái=0, lệch phải=0 -> lấy 7 (phải (hòa)); i=2, j=2, ans=[11, 8, 6, 6, 7]


In [19]:
_ = trace_two_pointers([1, 2, 3, 4, 5], 2)
print()
_ = trace_two_pointers([-7, 22, 17, 3], 4)

sắp: [1, 2, 3, 4, 5], tâm m = s[2] = 3
bước 1: lệch trái=2, lệch phải=2 -> lấy 5 (phải (hòa)); i=0, j=3, ans=[5]
bước 2: lệch trái=2, lệch phải=1 -> lấy 1 (trái); i=1, j=3, ans=[5, 1]

sắp: [-7, 3, 17, 22], tâm m = s[1] = 3
bước 1: lệch trái=10, lệch phải=19 -> lấy 22 (phải); i=0, j=2, ans=[22]
bước 2: lệch trái=10, lệch phải=14 -> lấy 17 (phải); i=0, j=1, ans=[22, 17]
bước 3: lệch trái=10, lệch phải=0 -> lấy -7 (trái); i=1, j=1, ans=[22, 17, -7]
bước 4: lệch trái=0, lệch phải=0 -> lấy 3 (phải (hòa)); i=1, j=0, ans=[22, 17, -7, 3]


## 9. Code nộp LeetCode

Bản khuyên dùng (hai con trỏ, một lần sắp), đúng chữ ký `getStrongest(self, arr: List[int], k: int) -> List[int]`:

In [20]:
class Solution:
    def getStrongest(self, arr: List[int], k: int) -> List[int]:
        arr.sort()
        m = arr[(len(arr) - 1) // 2]
        i, j = 0, len(arr) - 1
        ans = []
        for _ in range(k):
            if arr[j] - m >= m - arr[i]:      # hòa -> lấy bên phải (giá trị lớn hơn)
                ans.append(arr[j]); j -= 1
            else:
                ans.append(arr[i]); i += 1
        return ans

for a, k, want in examples:
    assert sorted(Solution().getStrongest(list(a), k)) == sorted(want)
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Khóa sắp xếp.** Theo đề, `x` mạnh hơn `y` ⇔ `|x−m| > |y−m|` hoặc (`=` và `x > y`) ⇔ `(|x−m|, x) > (|y−m|, y)` theo thứ tự từ điển. Đây là thứ tự toàn phần (trên giá trị), nên `k` phần tử đầu khi sắp giảm dần là `k` mạnh nhất.
2. **Hai con trỏ — phần tử mạnh nhất ở một đầu.** Mảng `s` tăng dần. Với mọi `t` trong `[i..j]`: nếu `s[t] ≥ m` thì `|s[t]−m| = s[t]−m ≤ s[j]−m`; nếu `s[t] < m` thì `|s[t]−m| = m−s[t] ≤ m−s[i]`. Nên phần tử có độ lệch lớn nhất là `s[i]` hoặc `s[j]`. Khi hòa độ lệch, `s[j] ≥ s[i]` nên `s[j]` mạnh hơn (hoặc bằng giá trị).
3. **Bỏ `abs` vẫn đúng.** Tâm `m` là một phần tử của `s`. Nếu `s[i] ≤ m ≤ s[j]` thì `s[j]−m` và `m−s[i]` chính là hai độ lệch. Nếu `s[j] < m` thì mọi phần tử còn lại đều `< m`, nên `s[i] < m`: vế trái âm, vế phải dương ⇒ lấy `s[i]` — đúng, vì khi mọi phần tử nằm bên trái tâm thì phần tử nhỏ nhất là xa tâm nhất. Nếu `s[i] > m` thì đối xứng: `m−s[i] < 0 < s[j]−m` ⇒ lấy `s[j]`, cũng đúng.
4. **Invariant:** mỗi bước lấy đúng phần tử mạnh nhất của phần còn lại, nên sau `k` bước có `k` phần tử mạnh nhất.

## 11. Phân tích Time / Space

Gọi `n = len(arr)`.

| Cách | Time | Space |
|---|---|---|
| 1. Chọn k lần | `O(n log n + n·k)` | `O(n)` |
| 2. Sắp theo khóa | `O(n log n)` (2 lần sort) | `O(n)` |
| 3. Hai con trỏ | `O(n log n + k)` (1 lần sort) | `O(k)` + bộ nhớ sort |
| 4. Quickselect + heap | trung bình `O(n + n log k)` | `O(n)` (bản sao) + `O(k)` |

Không có khác biệt lớn giữa trung bình và xấu nhất, trừ quickselect (xấu nhất `O(n²)`, rất hiếm với pivot ngẫu nhiên).

## 12. Bộ test

Vì thứ tự tùy ý, so `sorted(got)` với `sorted(want)`. Đối chiếu mọi cách với brute force trên mảng ngẫu nhiên (seed cố định, đúng ràng buộc giá trị), các edge case: `n = 1`, `k = n`, mọi phần tử bằng nhau, nhiều phần tử hòa độ lệch, số âm.

In [21]:
ALL = [get_strongest_brute, get_strongest_sort, get_strongest_two_pointers, get_strongest_heap,
       lambda a, k: Solution().getStrongest(list(a), k)]

def check_all(arr, k, want=None):
    res = [sorted(f(list(arr), k)) for f in ALL]
    if want is None:
        want = res[0]
    else:
        want = sorted(want)
    assert all(r == want for r in res), (arr, k, res)
    return want

for arr, k, want in examples:
    check_all(arr, k, want)

assert check_all([5], 1) == [5]
assert check_all([7, 7, 7, 7], 3) == [7, 7, 7]
assert check_all([1, 2, 3, 4], 2, [4, 3])                    # n chẵn: tâm = 2
assert check_all([-7, 22, 17, 3], 4, [22, 17, -7, 3])
assert check_all([-5, -1, 0, 1, 5], 2, [5, -5])              # hòa |5| = |-5| -> lấy 5 trước, rồi -5
assert check_all([-5, -1, 0, 1, 5], 1, [5])
assert check_all([-10**5, 10**5], 1, [10**5])                # tâm = -10^5, 10^5 xa hơn
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [22]:
rng = random.Random(2020)
for _ in range(2000):
    n = rng.randint(1, 30)
    lo = rng.choice([-5, -100, -10**5])
    arr = [rng.randint(lo, -lo) for _ in range(n)]
    k = rng.randint(1, n)
    check_all(arr, k)
print("Random 2000 test: OK")

Random 2000 test: OK


In [23]:
# Ràng buộc lớn: n = 10^5, k = n
rng = random.Random(8)
big = [rng.randint(-10**5, 10**5) for _ in range(10**5)]
t = time.perf_counter()
r = Solution().getStrongest(list(big), len(big))
print(f"n=10^5, k=n: {time.perf_counter() - t:.3f}s, 3 phần tử mạnh nhất: {r[:3]}")
assert sorted(r) == sorted(big)
assert sorted(Solution().getStrongest(list(big), 1000)) == sorted(get_strongest_sort(big, 1000))

n=10^5, k=n: 0.047s, 3 phần tử mạnh nhất: [99996, 99995, 99992]


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng trung vị trung bình cộng (`statistics.median`)

Đề định nghĩa tâm là phần tử ở `(n−1)//2`, không phải trung bình hai phần tử giữa.

In [24]:
def wrong_median(arr, k):
    m = statistics.median(arr)                        # SAI với n chẵn
    return sorted(arr, key=lambda x: (abs(x - m), x), reverse=True)[:k]

arr, k = [1, 2, 3, 4], 2
print("SAI :", wrong_median(arr, k), "| đúng:", get_strongest_sort(arr, k))
print("median = 2.5 nên 1 và 4 hòa; đúng ra tâm = 2 nên 3 (lệch 1, lớn hơn 1) mạnh hơn 1.")

SAI : [4, 1] | đúng: [4, 3]
median = 2.5 nên 1 và 4 hòa; đúng ra tâm = 2 nên 3 (lệch 1, lớn hơn 1) mạnh hơn 1.


**Sửa:** `m = sorted(arr)[(len(arr) - 1) // 2]`.

### Lỗi 2 — Lấy `arr[(n−1)//2]` khi chưa sắp xếp

In [25]:
def wrong_unsorted_centre(arr, k):
    m = arr[(len(arr) - 1) // 2]                      # SAI: arr chưa sắp
    return sorted(arr, key=lambda x: (abs(x - m), x), reverse=True)[:k]

arr, k = [6, -3, 7, 2, 11], 2
print("SAI :", wrong_unsorted_centre(arr, k), "(tâm lấy nhầm =", arr[2], ") | đúng:", get_strongest_sort(arr, k), "(tâm = 6)")

SAI : [-3, 2] (tâm lấy nhầm = 7 ) | đúng: [-3, 11] (tâm = 6)


**Sửa:** sắp trước rồi mới lấy chỉ số.

### Lỗi 3 — Chỉ số tâm `n // 2` thay vì `(n − 1) // 2`

Với `n` lẻ hai công thức trùng nhau, nhưng với `n` chẵn `n // 2` lấy phần tử giữa **bên phải**.

In [26]:
def wrong_index(arr, k):
    s = sorted(arr)
    m = s[len(s) // 2]                                # SAI với n chẵn
    return sorted(s, key=lambda x: (abs(x - m), x), reverse=True)[:k]

arr, k = [1, 2, 3, 4], 2
print("SAI :", sorted(wrong_index(arr, k)), "| đúng:", sorted(get_strongest_sort(arr, k)))

SAI : [1, 4] | đúng: [3, 4]


**Sửa:** `(len(s) - 1) // 2`.

### Lỗi 4 — Quên tie-break theo giá trị

Sắp chỉ theo `abs(x − m)` (sort ổn định giữ thứ tự ban đầu) ⇒ khi hòa có thể chọn phần tử **nhỏ hơn**.

In [27]:
def wrong_no_tiebreak(arr, k):
    s = sorted(arr)
    m = s[(len(s) - 1) // 2]
    return sorted(s, key=lambda x: -abs(x - m))[:k]   # SAI: thiếu tiêu chí x

arr, k = [1, 2, 3, 4, 5], 1
print("SAI :", wrong_no_tiebreak(arr, k), "| đúng:", get_strongest_sort(arr, k))

SAI : [1] | đúng: [5]


**Sửa:** khóa `(abs(x − m), x)` với `reverse=True` (hoặc `(-abs(x − m), -x)` không reverse).

### Lỗi 5 — Hai con trỏ dùng `>` thay cho `>=`

Khi hòa độ lệch phải lấy bên **phải** (giá trị lớn hơn). Dùng `>` sẽ lấy bên trái.

In [28]:
def wrong_strict(arr, k):
    s = sorted(arr)
    m = s[(len(s) - 1) // 2]
    i, j, ans = 0, len(s) - 1, []
    for _ in range(k):
        if s[j] - m > m - s[i]:                       # SAI: phải là >=
            ans.append(s[j]); j -= 1
        else:
            ans.append(s[i]); i += 1
    return ans

arr, k = [1, 2, 3, 4, 5], 1
print("SAI :", wrong_strict(arr, k), "| đúng:", get_strongest_two_pointers(arr, k))

SAI : [1] | đúng: [5]


**Sửa:** `if s[j] - m >= m - s[i]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- So sánh có tie-break ⇒ **khóa tuple** `(tiêu chí 1, tiêu chí 2, …)`; muốn giảm dần thì `reverse=True` hoặc đổi dấu từng thành phần.
- "Xa/gần một điểm `m` nhất" trên mảng **đã sắp** ⇒ **hai con trỏ từ hai đầu** (xa nhất) hoặc **cửa sổ quanh vị trí `m`** (gần nhất, bài 658).
- Chỉ cần `k` phần tử tốt nhất, `k ≪ n` ⇒ `heapq.nlargest/nsmallest`.
- Đề định nghĩa lại khái niệm quen thuộc (tâm, trung vị…) ⇒ **đọc kỹ công thức**, đừng dùng định nghĩa sách giáo khoa.
- Đáp án "thứ tự bất kỳ" ⇒ test bằng `sorted(...)`.

## 15. Biến thể của bài

1. **k giá trị yếu nhất:** đảo thứ tự khóa, hoặc hai con trỏ từ **tâm ra ngoài** (bài tập 3).
2. **658 — Find K Closest Elements:** k phần tử gần `x` nhất (hòa thì nhỏ hơn thắng), trả về đã sắp — cửa sổ trượt / tìm nhị phân trên vị trí bắt đầu cửa sổ (bài tập 5).
3. **Trả về theo đúng thứ tự từ mạnh đến yếu:** hai con trỏ đã cho sẵn thứ tự đó.
4. **Tâm là trung bình cộng** hoặc một giá trị `x` cho trước: chỉ đổi `m`, thuật toán giữ nguyên.
5. **Dòng dữ liệu (stream)** với `k` cố định: heap kích thước `k`, nhưng tâm thay đổi theo thời gian ⇒ cần hai heap để duy trì trung vị (bài 295) — khó hơn nhiều.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chọn k lần | Mỗi lượt quét tìm mạnh nhất | `O(n·k)` | `O(n)` | Kiểm chứng |
| 2. Sắp theo khóa | `key=(abs(x−m), x)`, `reverse=True` | `O(n log n)` | `O(n)` | Ngắn nhất, phỏng vấn nhanh |
| 3. Hai con trỏ | Mạnh nhất ở một trong hai đầu | `O(n log n + k)` | `O(k)` | Một lần sắp, thể hiện insight |
| 4. Quickselect + heap | Tâm `O(n)` + `nlargest` | `O(n + n log k)` | `O(n)` | `k` nhỏ, muốn tránh sắp toàn bộ |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `getStrongest([6,7,11,7,6,8], 2)` trả về những giá trị nào?

**Bài 2 (tính tay):** Với `arr = [-7, 22, 17, 3]`, viết thứ tự đầy đủ từ mạnh nhất đến yếu nhất.

**Bài 3 (điền TODO):** Viết `k_weakest(arr, k)` trả về `k` giá trị **yếu nhất** (yếu hơn = không mạnh hơn), thứ tự bất kỳ.

**Bài 4 (tự viết lại):** Viết lại hai con trỏ thành `strength_order(arr)` trả về **toàn bộ** mảng theo thứ tự mạnh → yếu, và kiểm tra với Cách 2.

**Bài 5 (mini challenge):** Viết `k_closest(arr, k, x)` (bài 658): `arr` đã sắp tăng, trả về `k` phần tử gần `x` nhất (hòa thì phần tử nhỏ hơn được ưu tiên), kết quả sắp tăng, độ phức tạp `O(log(n − k) + k)`.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def k_weakest(arr: List[int], k: int) -> List[int]:
    # TODO: sắp theo khóa (abs(x - m), x) TĂNG dần rồi lấy k phần tử đầu
    raise NotImplementedError

# Bài 4 — TODO
def strength_order(arr: List[int]) -> List[int]:
    # TODO: hai con trỏ chạy đủ n bước
    raise NotImplementedError

# Bài 5 — TODO
def k_closest(arr: List[int], k: int, x: int) -> List[int]:
    # TODO: tìm nhị phân vị trí bắt đầu cửa sổ lo trong [0, n-k]
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Tâm `m = 7`. (2) `11` lệch 4 mạnh nhất. (3) `8` và `6` cùng lệch 1, `8` lớn hơn.

**Bài 2.** (1) Sắp `[-7, 3, 17, 22]`, `m = 3`. (2) Độ lệch: `22→19, 17→14, -7→10, 3→0`. (3) `[22, 17, -7, 3]`.

**Bài 3.** (1) Yếu nhất = cuối danh sách mạnh → yếu. (2) Khóa `(abs(x−m), x)` tăng dần. (3) Cẩn thận hòa: phần tử **nhỏ hơn** yếu hơn.

**Bài 4.** (1) Copy Cách 3. (2) Đổi `range(k)` thành `range(n)`. (3) So với `sorted(arr, key=..., reverse=True)` — phải bằng nhau **cả thứ tự** (vì giá trị trùng nhau không phân biệt được).

**Bài 5.** (1) Cửa sổ kết quả là `arr[lo : lo + k]` liên tiếp. (2) So `x − arr[mid]` với `arr[mid + k] − x`: nếu bên trái xa hơn (`>`) thì dịch cửa sổ sang phải (`lo = mid + 1`), ngược lại `hi = mid`. (3) Tìm trên `lo ∈ [0, n − k]`.

## 19. Đáp án tham khảo cho bài tập

In [30]:
# Bài 1, Bài 2
r1 = get_strongest_two_pointers([6, 7, 11, 7, 6, 8], 2)
print("Bài 1:", r1)
assert sorted(r1) == [8, 11]
m2 = centre([-7, 22, 17, 3])
order2 = sorted([-7, 22, 17, 3], key=lambda x: (abs(x - m2), x), reverse=True)
print("Bài 2:", order2)
assert order2 == [22, 17, -7, 3]

Bài 1: [11, 8]
Bài 2: [22, 17, -7, 3]


In [31]:
def k_weakest_ans(arr, k):
    m = centre(arr)
    return sorted(arr, key=lambda x: (abs(x - m), x))[:k]

def strength_order_ans(arr):
    s = sorted(arr)
    m = s[(len(s) - 1) // 2]
    i, j, out = 0, len(s) - 1, []
    while i <= j:
        if s[j] - m >= m - s[i]:
            out.append(s[j]); j -= 1
        else:
            out.append(s[i]); i += 1
    return out

assert sorted(k_weakest_ans([1, 2, 3, 4, 5], 2)) == [2, 3]
rng = random.Random(6)
for _ in range(500):
    arr = [rng.randint(-20, 20) for _ in range(rng.randint(1, 25))]
    k = rng.randint(1, len(arr))
    order = sorted(arr, key=lambda x: (abs(x - centre(arr)), x), reverse=True)
    assert strength_order_ans(arr) == order
    assert sorted(k_weakest_ans(arr, k)) == sorted(order[len(arr) - k:])
print("Bài 3, 4: OK", strength_order_ans([6, 7, 11, 7, 6, 8]))

Bài 3, 4: OK [11, 8, 6, 6, 7, 7]


In [32]:
def k_closest_ans(arr, k, x):
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        if x - arr[mid] > arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo:lo + k]

def k_closest_brute(arr, k, x):
    return sorted(sorted(arr, key=lambda v: (abs(v - x), v))[:k])

assert k_closest_ans([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert k_closest_ans([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
rng = random.Random(10)
for _ in range(1000):
    arr = sorted(rng.randint(-15, 15) for _ in range(rng.randint(1, 20)))
    k = rng.randint(1, len(arr))
    x = rng.randint(-20, 20)
    assert k_closest_ans(arr, k, x) == k_closest_brute(arr, k, x), (arr, k, x)
print("Bài 5: OK", k_closest_ans([1, 2, 3, 4, 5], 4, 3))

Bài 5: OK [1, 2, 3, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
arr.sort()
m = arr[(len(arr) - 1) // 2]          # tâm theo đề
i, j, ans = 0, len(arr) - 1, []
for _ in range(k):
    if arr[j] - m >= m - arr[i]:      # >= : hòa thì lấy giá trị lớn
        ans.append(arr[j]); j -= 1
    else:
        ans.append(arr[i]); i += 1
return ans
# hoặc: sorted(arr, key=lambda x: (abs(x - m), x), reverse=True)[:k]
```

**Checklist:**
- [ ] Tâm lấy **sau khi sắp**, chỉ số `(n − 1) // 2`.
- [ ] Không dùng `statistics.median`.
- [ ] Có tie-break theo giá trị (lớn hơn thì mạnh hơn).
- [ ] Hai con trỏ dùng `>=`.
- [ ] Đã thử `n = 1`, `k = n`, mảng toàn giá trị bằng nhau, `n` chẵn.